# ZOD02 — trusted 12-checkpoint cascade

This notebook applies the exact 22-rule prefix from the full frozen V0 precision ranking, stopping before the 13th distinct checkpoint. It uses the checkpoint with the highest V0 accuracy among zero-overlap eligible checkpoints as fallback (N0089). V0 is used for rule selection, so these full-V0 results are in-sample diagnostics.


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from src.zod02 import _validation_frame, _load_cached_candidates, _metrics, _apply_fixed_zod01
from src.zod01 import DEFAULT_MINIMUM_SUPPORT, DEFAULT_PRIOR_ALPHA, DEFAULT_PRIOR_BETA
catalog = pd.read_csv("docs/checkpoint_label_catalog.csv", dtype=str, keep_default_na=False).drop_duplicates("experiment_id").set_index("experiment_id")

validation = _validation_frame()
classes = sorted(validation.y_true.unique().tolist())
package = json.loads(Path("artifacts/kaggle/zod02-models/zod02_manifest.json").read_text())
fallback_id = str(package["fallback_candidate_id"])
package_candidate_ids = {entry["candidate_id"] for entry in package["candidates"].values()}
_, all_top1, metadata, zod01_rules, exclusions, _ = _load_cached_candidates(
    validation=validation,
    inventory_path="artifacts/analysis/zod01_specialist_discovery/checkpoint_inventory.csv",
    ranked_rules_path="artifacts/analysis/zod01_specialist_discovery/ranked_checkpoint_label_rules.csv",
    predictions_dir="artifacts/analysis/zod01_specialist_discovery/predictions",
    fallback_id=fallback_id)
missing = package_candidate_ids - set(all_top1)
assert not missing, f"packaged candidate prediction missing: {missing}"
top1_by_id = {candidate_id: all_top1[candidate_id] for candidate_id in package_candidate_ids}
y = validation.y_true.to_numpy(dtype=str)


In [ ]:
trust = pd.read_csv("artifacts/analysis/zod02/trust_list.csv")
trust["rank"] = pd.to_numeric(trust["rank"], errors="raise").astype(int)
trust = trust.sort_values("rank", kind="stable").reset_index(drop=True)
assert len(trust) == 22
assert trust.sequence_id.nunique() == 12
assert set(trust.candidate_id).issubset(package_candidate_ids)
assert [r["sequence_id"] for r in package["rules"]] == trust.sequence_id.tolist()
assert [r["label"] for r in package["rules"]] == trust.label.tolist()
rules = [{"order": int(row.rank), "candidate_id": str(row.candidate_id),
          "sequence_id": str(row.sequence_id), "label": str(row.label),
          "precision": float(row.precision),
          "smoothed_precision": float(row.smoothed_precision),
          "prediction_count": int(row.prediction_count)}
         for row in trust.itertuples()]
assigned = np.zeros(len(y), dtype=bool)
prediction = np.empty(len(y), dtype=object)
selected_candidate = np.empty(len(y), dtype=object)
trace = []
for rule in rules:
    cid, label = rule["candidate_id"], rule["label"]
    rows = np.flatnonzero((~assigned) & (top1_by_id[cid] == label))
    assigned[rows] = True
    prediction[rows] = label
    selected_candidate[rows] = cid
    correct = int(np.sum(y[rows] == label))
    trace.append({**rule, "assigned_examples": int(len(rows)),
                  "incremental_true_positives": correct,
                  "incremental_false_positives": int(len(rows)-correct),
                  "remaining_rows": int((~assigned).sum())})
remaining = np.flatnonzero(~assigned)
fallback_id = str(package["fallback_candidate_id"])
prediction[remaining] = all_top1[fallback_id][remaining]
selected_candidate[remaining] = fallback_id
trace.append({"order": "fallback", "candidate_id": fallback_id,
              "sequence_id": package["fallback_sequence_id"], "label": "checkpoint top-1",
              "assigned_examples": int(len(remaining)),
              "incremental_true_positives": int(np.sum(y[remaining] == prediction[remaining])),
              "incremental_false_positives": int(np.sum(y[remaining] != prediction[remaining])),
              "remaining_rows": 0})
zod01_prediction, _ = _apply_fixed_zod01(all_top1, y, zod01_rules, fallback_id)
metrics = {"zod02_full_validation": _metrics(y, prediction, classes),
           "zod01_full_validation": _metrics(y, zod01_prediction, classes),
           "corrected_vs_zod01": int(np.sum((prediction == y) & (zod01_prediction != y))),
           "regressed_vs_zod01": int(np.sum((prediction != y) & (zod01_prediction == y))),
           "rules": len(rules), "candidate_count": len(package_candidate_ids),
           "fallback_id": fallback_id,
           "selection_caveat": "Trust rules are ranked on the complete V0 validation labels; this is an in-sample selection score."}
metrics["delta_accuracy"] = metrics["zod02_full_validation"]["accuracy"] - metrics["zod01_full_validation"]["accuracy"]
metrics["delta_balanced_accuracy"] = metrics["zod02_full_validation"]["balanced_accuracy"] - metrics["zod01_full_validation"]["balanced_accuracy"]
predictions = pd.DataFrame({"id":validation.id, "prompt_id":validation.prompt_id, "y_true":y,
    "zod01_prediction":zod01_prediction, "zod02_prediction":prediction,
    "selected_candidate_id":selected_candidate, "corrected_by_zod02":(prediction==y)&(zod01_prediction!=y),
    "regressed_by_zod02":(prediction!=y)&(zod01_prediction==y)})
out = Path("artifacts/analysis/zod02_precision_ranked"); out.mkdir(parents=True, exist_ok=True)
pd.DataFrame(trace).to_csv(out/"rule_application_full_v0.csv", index=False)
predictions.to_parquet(out/"predictions_full_v0.parquet", index=False)
(out/"metrics.json").write_text(json.dumps(metrics, indent=2)+"\n")
result = {"metrics":metrics, "rules":rules, "trace":pd.DataFrame(trace), "predictions":predictions}
print(json.dumps({k:v for k,v in metrics.items() if k not in {"zod02_full_validation","zod01_full_validation"}}, indent=2))


In [ ]:
comparison = pd.DataFrame([
 {"Cascade":"ZOD01","Accuracy":metrics["zod01_full_validation"]["accuracy"],"Balanced accuracy":metrics["zod01_full_validation"]["balanced_accuracy"],"Correct":f'{metrics["zod01_full_validation"]["correct"]}/540'},
 {"Cascade":"ZOD02","Accuracy":metrics["zod02_full_validation"]["accuracy"],"Balanced accuracy":metrics["zod02_full_validation"]["balanced_accuracy"],"Correct":f'{metrics["zod02_full_validation"]["correct"]}/540'}])
comparison["Delta vs ZOD01"]=[0,metrics["delta_balanced_accuracy"]]
display(comparison.style.format({"Accuracy":"{:.2%}","Balanced accuracy":"{:.2%}","Delta vs ZOD01":"{:+.2%}"}))
print(metrics["selection_caveat"])

In [ ]:
display(result["trace"].style.format({"precision":"{:.1%}","smoothed_precision":"{:.1%}"}))

In [ ]:
z1,z2=metrics["zod01_full_validation"]["per_label"],metrics["zod02_full_validation"]["per_label"]
per_label=pd.DataFrame([{"Label":l,"ZOD01 recall":z1[l]["recall"],"ZOD02 recall":z2[l]["recall"],"Correct ZOD01":z1[l]["true_positive"],"Correct ZOD02":z2[l]["true_positive"],"Delta":z2[l]["recall"]-z1[l]["recall"]} for l in z1])
display(per_label.style.format({"ZOD01 recall":"{:.1%}","ZOD02 recall":"{:.1%}","Delta":"{:+.1%}"}))

In [ ]:
changed=result["predictions"].loc[result["predictions"].corrected_by_zod02|result["predictions"].regressed_by_zod02]
print("Corrected:",metrics["corrected_vs_zod01"],"Regressed:",metrics["regressed_vs_zod01"],"Changed rows:",len(changed))
display(changed.sort_values(["corrected_by_zod02","id"],ascending=[False,True]))